# DBT Wrapper

DBT Wrapper that runs DBT in stages, while sending monitoring logs in between.

As the Monitoring library is in Scala, it requires calling Scala code from Python.

#### Parameters

* **dbt_project**: Path to the the DBT project.
* **model_paths**: Paths to the models to run in order. Defaults to ["silver", "gold"].
* **target**: Target profile to use in DBT.
* **properties_file**: Path to the properties file containing the Monitoring Exporter properties.
* **job_name**: Name of the job for the monitoring logs.
* **job_id**: Id of the job for the monitoring logs.
* **run_id**: Id of the run for the monitoring logs.
* **extra_flags**: Map of extra flags that may be needed for DBT run where the keys must be values of model_paths. Any value of model_paths not present in the map will run without any extra flags. Defaults to {}.
* **tags**: Map of tags to send on the log. Defaults to {}.

### Install DBT

Installs the required DBT libraries and re-starts python to allow for their use.

In [0]:
%pip install dbt-core dbt-databricks databricks-sql-connector

In [0]:
dbutils.library.restartPython()

### Set-Up DBT

Sets up the DBT models to run and the DBT project directory based on the Notebook parameters 

In [0]:
import json
import os

# Get DBT models to run
dbutils.widgets.text("model_paths", "[]")
model_paths = json.loads(dbutils.widgets.get("model_paths"))

if len(model_paths) == 0:
    model_paths = ["silver", "gold"]

dbutils.widgets.text("extra_flags", "{}")
extra_flags = json.loads(dbutils.widgets.get("extra_flags"))

dbutils.widgets.text("tags", "{}")
tags = json.loads(dbutils.widgets.get("tags"))

# Set-up DBT environmental variables
os.environ["DBT_PROJECT_DIR"] = dbutils.widgets.get("dbt_project")
os.environ["DBT_PROFILES_DIR"] = dbutils.widgets.get("dbt_project")

In [0]:
from datetime import datetime, timedelta

for model_name, flags in extra_flags.items():
    vars_dict = flags.get("vars", {})
    exec_date = vars_dict.get("exec_date")
    
    if exec_date == "" or exec_date is None:
        # Data do dia anterior
        yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
        vars_dict["exec_date"] = yesterday
        flags["vars"] = vars_dict 
        extra_flags[model_name] = flags
print(extra_flags)

### Set-Up the Monitoring Exporter

Initializes the Monitoring Exporter and loads the Scala classes necessary to use it with Python

In [0]:
%scala
import com.brisa.monitoring.MonitoringExporter
import com.brisa.monitoring.model.{Log, LogStatusType}

val propertiesFile = dbutils.widgets.get("properties_file")
MonitoringExporter.initialize(propertiesFile, "DBT", dbutils.widgets.get("job_name"), dbutils.widgets.get("job_id"), dbutils.widgets.get("run_id"), None, None)

In [0]:
jvm = spark._jvm
HashMap = jvm.scala.collection.mutable.HashMap
LocalDateTime = jvm.java.time.LocalDateTime
ZoneOffset = jvm.java.time.ZoneOffset
Log = jvm.com.brisa.monitoring.model.Log
LogStatusType = jvm.com.brisa.monitoring.model.LogStatusType
MonitoringExporter = jvm.com.brisa.monitoring.MonitoringExporter

### Utils

Util functions

In [0]:
from dbt.cli.main import dbtRunnerResult

def create_successful_models_metrics(runner_result: dbtRunnerResult):
    """
    Creates the metrics for the successful models, namely the elapsed time in seconds.
    """
    metrics = HashMap()
    for model_result in runner_result.result.results:
        if model_result.status.value == "success":
            time = HashMap()
            time.put("elapsed", model_result.execution_time)
            metrics.put(model_result.node.name, time)
    return metrics

def create_successful_metrics(runner_result: dbtRunnerResult):
    """
    Creates the metrics for a successful DBT run
    """
    metrics = HashMap()
    successful_models = create_successful_models_metrics(runner_result)
    metrics.put("successful_models", successful_models)
    return metrics

def create_failure_metrics(runner_result: dbtRunnerResult):
    """
    Creates the metrics for a failed DBT run
    """
    metrics = HashMap()
    if runner_result.result:
        successful_models = create_successful_models_metrics(runner_result)
        metrics.put("successful_models", successful_models)

        failed_models = [model_result.node.name for model_result in runner_result.result.results if model_result.status.value != "success"]
        metrics.put("failed_models", failed_models)

        errors = HashMap()
        for model_result in runner_result.result.results:
            if model_result.status.value != "success":
                if model_result.message:
                    errors.put(model_result.node.name, model_result.message[:200])

        metrics.put("errors", errors)
    elif runner_result.exception:
        metrics.put("errors", str(runner_result.exception)[:1000])
    else:
        metrics.put("errors", "Unknown errors")

    return metrics

### Main

Set-up required variables and run DBT once per model_path, sending a monitoring log before and after each run.

In [0]:
from dbt.cli.main import dbtRunner

tool = "DBT"
process = dbutils.widgets.get("job_name")
job_id = dbutils.widgets.get("job_id")
run_id = dbutils.widgets.get("run_id")
target = dbutils.widgets.get("target")
checkpoint_file_path = dbutils.widgets.get("checkpoint_file_path")
if not checkpoint_file_path:
    raise Exception("checkpoint_file_path parameter is required")
extra_info = HashMap()

execution_arguments = HashMap()
execution_arguments.put("properties_file", dbutils.widgets.get("properties_file"))
execution_arguments.put("model_paths", model_paths)
execution_arguments.put("target", target)
execution_arguments.put("dbt_project", dbutils.widgets.get("dbt_project"))
execution_arguments.put("extra_flags", extra_flags)
execution_arguments.put("checkpoint_file_path", checkpoint_file_path)

scala_tags = HashMap()
for key, value in tags.items():
    scala_tags.put(key, value)

dbt_runner = dbtRunner()

In [0]:
def get_last_successful_date(file_path):

    if not os.path.exists(file_path):
        return None

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            first_line = f.readline().strip()

            if first_line:
                return datetime.strptime(first_line, "%Y-%m-%d").date()

    except Exception as e:
        print(f"Warning reading tracking file: {e}")

    return None

In [0]:
def generate_date_range(start_date, end_date):
    current = start_date
    while current <= end_date:
        yield current
        current += timedelta(days=1)


In [0]:
import os

def update_checkpoint_file(file_path, execution_date):
    execution_date_str = execution_date.strftime("%Y-%m-%d")

    current_date = None
    # Ler data atual (se existir)
    if os.path.exists(file_path):
        try:
            with open(file_path, "r") as f:
                first_line = f.readline().strip()
                if first_line:
                    current_date = datetime.strptime(first_line, "%Y-%m-%d").date()
        except Exception as e:
            print(f"Warning reading checkpoint file: {e}")

    # Só atualizar se:
    # - não existir data atual
    # - ou nova data for mais recente
    if current_date is None or execution_date > current_date:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)

        with open(file_path, "w") as f:
            f.write(execution_date_str + "\n")

        print(f"Checkpoint file updated with date: {execution_date_str}")
    else:
        print(f"Checkpoint file NOT updated. Current date ({current_date}) is more recent.")

In [0]:
VOLUME_BASE_PATH = "/Volumes/operational_dev/volumes/modelling_tools/"
checkpoint_file = VOLUME_BASE_PATH + checkpoint_file_path

for model_path in model_paths:
    metrics = HashMap()
    running_log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, model_path, LogStatusType.RUNNING, metrics, extra_info, f"Running models from '{model_path}'", LocalDateTime.now(ZoneOffset.UTC))
    MonitoringExporter.sendLog(running_log)

    try:
        dates_to_run = [None] 

        # Verifica se existe exec_date nas flags
        if model_path in extra_flags and "vars" in extra_flags[model_path]:
            vars_dict = extra_flags[model_path]["vars"]
            exec_date_str = vars_dict.get("exec_date")
            print(exec_date_str)

            if exec_date_str:
                start_date = datetime.strptime(exec_date_str, "%Y-%m-%d").date()
                end_date = get_last_successful_date(checkpoint_file)
                print(start_date)
                print(end_date)
                if end_date is None:
                    print("No checkpoint found - first execution")

                    if exec_date_str:
                        start_date = datetime.strptime(exec_date_str, "%Y-%m-%d").date()
                        dates_to_run = [start_date]
                    else:
                        # fallback: ontem
                        yesterday = datetime.now().date() - timedelta(days=1)
                        dates_to_run = [yesterday]
                if end_date:
                    if start_date > end_date:
                        #Catchup
                        backfill_start = end_date + timedelta(days=1)
                        backfill_end = start_date
                        dates_to_run = list(generate_date_range(backfill_start, backfill_end))
                    else:
                        dates_to_run = [start_date]
        print(dates_to_run)
        overall_success = True

        for date_to_run in dates_to_run:

            commands = ["run", "--target", target, "--select", model_path]

            if model_path in extra_flags:
                flags = extra_flags[model_path]

                # Atualiza exec_date
                if date_to_run and "vars" in flags:
                    flags["vars"]["exec_date"] = date_to_run.strftime("%Y-%m-%d")

                for key, value in flags.items():
                    commands.append(f"--{key}")
                    if isinstance(value, (dict, list)):
                        commands.append(json.dumps(value))
                    elif value != "":
                        commands.append(str(value))

            result = dbt_runner.invoke(commands)

            if not result.success:
                overall_success = False
                break

        if overall_success:
            if dates_to_run and dates_to_run[0] is not None:
                last_successful_date = dates_to_run[-1]
            elif len(dates_to_run) == 1 and dates_to_run[0] is not None:
                last_successful_date = dates_to_run[0]
            else:
                last_successful_date = None
                
            if last_successful_date:
                update_checkpoint_file(
                    checkpoint_file,
                    last_successful_date
                )
            status = LogStatusType.SUCCESSFUL
            message = f"Ran models from '{model_path}' successfully"
            metrics = create_successful_metrics(result)
        else:
            status = LogStatusType.FAILED
            message = f"Failed to run models from '{model_path}'"
            metrics = create_failure_metrics(result)

        log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, model_path, status, metrics, extra_info, message, LocalDateTime.now(ZoneOffset.UTC))
        MonitoringExporter.sendLog(log)

    except Exception as exception:
        status = LogStatusType.FAILED
        message = f"Unexpected failure when running models from '{model_path}'"
        metrics.put("errors", str(exception)[:1000])

        log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, model_path, status, metrics, extra_info, message, LocalDateTime.now(ZoneOffset.UTC))
        MonitoringExporter.sendLog(log)

        raise exception

    if status == LogStatusType.FAILED:
        raise Exception(f"Failed during '{model_path}' models")

In [0]:
exec_date = None

for flags in extra_flags.values():
    vars_dict = flags.get("vars", {})
    if "exec_date" in vars_dict:
        exec_date = vars_dict["exec_date"]
        break

commands = ["test", "--target", target]

if exec_date:
    commands.extend(["--vars", json.dumps({"exec_date": exec_date})])

result = dbt_runner.invoke(commands)

In [0]:
dbutils.notebook.run("/Workspace/BAE/DAD/gold/dbt-modelling/load_dbt_test_results", 0, {"target": target})

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7847911744546812>, line 1
----> 1 dbutils.notebook.run("load_dbt_test_results", 0, {"target": target})

NameError: name 'target' is not defined